In [1]:
!pip install pygad

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 2.9 MB/s eta 0:00:00


In [2]:
import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns
import pygad
import numpy as np

In [3]:
# Lista de símbolos de acciones
symbols = ['MELI', 'AMZN', "AAPL", "TSLA", "SPY", "GDX", "KO"]
market = "SPY"
time = "12mo"

In [4]:
# Get the ticker data
name = []
betas = []
for i in symbols:
    try:
        tickerData = yf.Ticker(i)
        data = yf.download(i, period=time)
        if data.empty:
            print(f"No data found for {i}. Skipping...")
            continue
        prec_i = data['Adj Close']
        returns_i = np.log(prec_i / prec_i.shift(1)).dropna()
        prec_m = yf.download(market, period=time)['Adj Close']
        if prec_m.empty:
            print(f"No data found for {market}. Skipping...")
            continue

        returns_m = np.log(prec_m / prec_m.shift(1)).dropna()
        if returns_i.empty or returns_m.empty:
            print(f"Empty returns for {i} or {market}. Skipping covariance calculation...")
            continue

        cov_i_m = np.cov(returns_i, returns_m)[0][1]  # covariance between stock and market
        var_m = np.var(returns_m)  # variance of market
        beta = (cov_i_m / var_m).round(2)
        med_i = (np.mean(returns_i) * 252)
        std_i = (np.std(returns_i) * np.sqrt(252))
        name.append([i, f"Media: {med_i:.2%}", f"Volatilidad: {std_i:.2%}", f"Beta: {beta:.2f}"])
        betas.append(beta)
    except Exception as e:
        print(f'Error processing {i}: {e}')

print(name)
print(betas)

[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MELI']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


No data found for MELI. Skipping...


[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AMZN']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


No data found for AMZN. Skipping...


[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


...

No data found for SPY. Skipping...


[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['GDX']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


No data found for GDX. Skipping...


[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['KO']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


No data found for KO. Skipping...
[]
[]


In [ ]:
# Descarga de los precios históricos
prices = yf.download(symbols, period=time)['Adj Close']
spy = yf.download(tickers='SPY', period=time)['Adj Close']

# Retornos diarios
returns = np.log(prices / prices.shift(1)).dropna()
return_spy = np.log(spy / spy.shift(1)).dropna()

returns


In [ ]:
# Matriz de covarianza
cov_matrix = returns.cov()

# Matriz de correlación
corr_matrix = returns.corr()

# Configurar tamaño de la figura
plt.figure(figsize=(5, 5))

# Heatmap
sns.heatmap(corr_matrix, annot=True, cmap='YlGn')
plt.title('Matriz de Correlación - Retornos diarios')
#plt.savefig('correlation_matrix.png')
plt.show()

In [ ]:
num_portfolios = 50
risk_free_rate = 0.035

def fitness_func(ga_instance, solution, solution_idx):
    weights = np.array(solution)
    portfolio_return = np.sum(returns.mean() * weights) * 252 # LA BOLSA OPERA 252 DÍAS / AÑO
    portfolio_std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
    # Sharpe Ratio - fitness function
    sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_std_dev

    return sharpe_ratio


In [ ]:
# GA Configuration
num_generations = 10000
num_parents_mating = 10
initial_population = np.random.uniform(low=0.01, high=0.4, size=(num_portfolios, len(symbols)))
initial_population = initial_population / initial_population.sum(axis=1, keepdims=True)



ga_instance = pygad.GA(num_generations=num_generations,
                       num_parents_mating=num_parents_mating,
                       fitness_func=fitness_func,
                       sol_per_pop=num_portfolios,
                       mutation_probability=0.6,
                       num_genes=len(symbols),
                       gene_space=[0.01, 0.4],
                       initial_population=initial_population,
                       save_solutions=True)

initial_population


In [ ]:
ga_instance.run()

In [ ]:
ga_instance.plot_fitness()
ga_instance.plot_genes()
ga_instance.plot_new_solution_rate()
ga_instance.plot_population()
ga_instance.plot_survival()

In [ ]:
# Extract results
solution, solution_fitness, _ = ga_instance.best_solution()
optimal_weights = np.array(solution)
optimal_weights /= np.sum(optimal_weights)  # normalize to make the sum 1
tickers_weights = dict(zip(symbols, np.round(optimal_weights, 2)))

print("Optimal Weights: ", tickers_weights)

In [ ]:
ga_instance.best_solution()

In [ ]:
solution_fitness

In [ ]:
mean_return = return_spy.mean() * 252
std_dev_return = (return_spy.std()) * np.sqrt(252)
sharpe_ratio_spy = (mean_return - risk_free_rate) / std_dev_return
print(mean_return)
print(std_dev_return)
print(sharpe_ratio_spy)

In [ ]:
ga_instance.best_solutions_fitness

In [ ]:
ga_instance.best_solution_generation